# 01 — Full-Text Extraction and Comparative Metric Mining

Purpose: mine candidate metrics from the recovered source texts for data centres and comparator assets. This notebook produces candidate rows only. Every row must be checked against the source quote/page before entering the final comparative coding sheet.


In [ ]:
# Optional first-time setup:
# %pip install -q pandas openpyxl

from pathlib import Path
import re, json, csv, math
import pandas as pd


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "Media_Article").is_dir() and (p / "Research_Workflow").is_dir():
            return p
    raise RuntimeError("Could not locate DC_job_potential root. Run this notebook inside the project folder.")

ROOT = find_project_root(Path.cwd())
EXTRACTION = ROOT / "Research_Workflow" / "02_Source_Extraction"
DATA_DIR = EXTRACTION / "data"
TEXT_DIR = DATA_DIR / "fulltext"
REGISTER_DIR = DATA_DIR / "source_register"
OUT_DIR = DATA_DIR / "coding_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

status_path = OUT_DIR / "source_recovery_status.csv"
text_inv_path = OUT_DIR / "fulltext_inventory.csv"
source_register_path = REGISTER_DIR / "source_leads_seed.csv"
comparative_template_path = REGISTER_DIR / "comparative_asset_coding_template.csv"

print("ROOT:", ROOT)
print("TEXT_DIR:", TEXT_DIR)
print("text inventory exists:", text_inv_path.exists())


In [ ]:
sources = pd.read_csv(source_register_path) if source_register_path.exists() else pd.DataFrame()
text_inv = pd.read_csv(text_inv_path) if text_inv_path.exists() else pd.DataFrame(columns=["source_id", "text_file", "words"])
print("sources:", len(sources), "texts:", len(text_inv))
text_inv.head()


In [ ]:
# Metric pattern library.
# The extractor is deliberately recall-oriented. Precision comes from RA/researcher verification.
NUM = r"(?P<num>\d{1,3}(?:,\d{3})*(?:\.\d+)?|\d+(?:\.\d+)?)"
MONEY = r"(?P<money>(?:A\$|US\$|USD|AUD|\$)\s?\d{1,3}(?:,\d{3})*(?:\.\d+)?\s?(?:billion|bn|million|m)?)"

PATTERNS = [
    ("construction_jobs_peak", re.compile(NUM + r"\s*(?:peak\s*)?(?:construction\s*)?(?:workers?|jobs?|FTEs?)", re.I), ["construction", "build", "built", "site", "peak"]),
    ("operational_jobs_fte", re.compile(NUM + r"\s*(?:permanent|operational|operations|ongoing|direct)?\s*(?:jobs?|workers?|FTEs?|staff)", re.I), ["operation", "operational", "permanent", "ongoing", "staff"]),
    ("jobs_per_mw", re.compile(NUM + r"\s*(?:jobs?|workers?|FTEs?)\s*(?:per|/)\s*MW", re.I), ["mw", "megawatt"]),
    ("fte_per_mw", re.compile(NUM + r"\s*FTEs?\s*(?:per|/)\s*MW", re.I), ["mw", "operation", "operational"]),
    ("construction_duration_months", re.compile(NUM + r"\s*(?:to\s*\d+\s*)?(?:months?|month)\b", re.I), ["construction", "build", "duration", "phase"]),
    ("construction_duration_years", re.compile(NUM + r"\s*(?:to\s*\d+\s*)?(?:years?|yr)\b", re.I), ["construction", "build", "duration", "phase"]),
    ("construction_job_years", re.compile(NUM + r"\s*(?:job[- ]?years?|person[- ]?years?|FTE[- ]?years?)", re.I), ["construction", "build", "job-year", "person-year", "fte-year"]),
    ("capex", re.compile(MONEY, re.I), ["capex", "capital", "investment", "cost", "project value", "development"]),
    ("capacity_mw", re.compile(NUM + r"\s*(?:MW|megawatts?)\b", re.I), ["mw", "megawatt", "capacity", "load"]),
    ("gfa_m2", re.compile(NUM + r"\s*(?:m2|m²|sqm|square metres|square meters)\b", re.I), ["gfa", "gross floor", "floor area", "building"]),
    ("gfa_sqft", re.compile(NUM + r"\s*(?:sq\.?\s*ft|square feet|sf)\b", re.I), ["gfa", "gross floor", "floor area", "building"]),
    ("io_multiplier", re.compile(NUM + r"\s*(?:x|times|multiplier)\b", re.I), ["multiplier", "input-output", "implan", "indirect", "induced"]),
    ("causal_percent_effect", re.compile(NUM + r"\s*%", re.I), ["employment", "wage", "effect", "increase", "synthetic", "difference-in-differences", "DiD"]),
]

PHASE_TERMS = {
    "construction": ["construction", "build", "built", "site", "contractor", "trades", "mep", "commissioning"],
    "operations": ["operation", "operational", "permanent", "ongoing", "staff", "facility management"],
    "lifecycle_or_economy": ["indirect", "induced", "supported", "multiplier", "economy", "county", "regional"],
}

def parse_num(s):
    try:
        return float(str(s).replace(",", ""))
    except Exception:
        return None

def classify_phase(text):
    low = text.lower()
    scores = {k: sum(1 for term in terms if term.lower() in low) for k, terms in PHASE_TERMS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] else "unspecified"

def sentence_split(text):
    # Keep page markers as anchors and split conservatively.
    parts = re.split(r"(?<=[.!?])\s+|\n{2,}", text)
    return [p.strip() for p in parts if p and p.strip()]

def current_page(sentence, previous):
    m = re.search(r"\[\[PAGE\s+(\d+)\]\]", sentence)
    return m.group(1) if m else previous


In [ ]:
# Mine candidate rows.
source_meta = sources.set_index("source_id").to_dict("index") if not sources.empty else {}
rows = []

for _, ti in text_inv.iterrows():
    source_id = ti["source_id"]
    text_path = Path(ti["text_file"])
    if not text_path.exists():
        continue
    text = text_path.read_text(encoding="utf-8", errors="ignore")
    page = ""
    for sent in sentence_split(text):
        page = current_page(sent, page)
        low = sent.lower()
        if not re.search(r"\d", sent):
            continue
        # Skip references/citation-heavy lines unless they contain a clear metric word.
        if not any(w in low for w in ["job", "employ", "worker", "fte", "mw", "megawatt", "capex", "investment", "billion", "million", "m2", "m²", "sq", "multiplier", "%"]):
            continue
        for metric, rx, context_terms in PATTERNS:
            if context_terms and not any(t.lower() in low for t in context_terms):
                continue
            for m in rx.finditer(sent):
                val = m.groupdict().get("num") or m.groupdict().get("money") or m.group(0)
                meta = source_meta.get(source_id, {})
                rows.append({
                    "source_id": source_id,
                    "asset_type": meta.get("asset_type", ""),
                    "source_group": meta.get("source_group", ""),
                    "title": meta.get("title", ""),
                    "metric": metric,
                    "raw_value": val,
                    "numeric_value": parse_num(val),
                    "phase_guess": classify_phase(sent),
                    "page_or_location": page,
                    "context_quote": re.sub(r"\s+", " ", sent)[:700],
                    "text_file": str(text_path),
                    "verification_status": "candidate_unverified",
                })

cand = pd.DataFrame(rows)
out = OUT_DIR / "candidate_metric_extractions.csv"
cand.to_csv(out, index=False)
print("Wrote", out, "rows", len(cand))
cand.groupby(["asset_type", "metric"]).size().sort_values(ascending=False).head(30) if not cand.empty else "No candidate rows mined"


In [ ]:
# Create or refresh a researcher-facing comparative coding workbook-style CSV.
# This starts from the template and adds suggested source rows where possible.
template_cols = list(pd.read_csv(comparative_template_path, nrows=0).columns)

if not cand.empty:
    # One placeholder row per source with any mined candidates. Researcher/RA fills verified fields.
    placeholder = []
    for source_id, g in cand.groupby("source_id"):
        meta = source_meta.get(source_id, {})
        placeholder.append({
            "record_id": f"{source_id}_001",
            "source_id": source_id,
            "asset_type": meta.get("asset_type", ""),
            "project_or_source_name": meta.get("title", ""),
            "location": "",
            "country": "",
            "year": meta.get("year", ""),
            "source_type": meta.get("source_group", ""),
            "method": "",
            "funding_or_bias_flag": "",
            "source_url": meta.get("url", ""),
            "full_text_file": str(g.iloc[0]["text_file"]),
            "confidence": "unverified_candidate",
            "ra_verification_status": "not_started",
        })
    coding = pd.DataFrame(placeholder)
else:
    coding = pd.DataFrame(columns=template_cols)

for col in template_cols:
    if col not in coding.columns:
        coding[col] = ""
coding = coding[template_cols]

csv_path = OUT_DIR / "comparative_asset_coding_sheet_draft.csv"
xlsx_path = OUT_DIR / "comparative_asset_coding_sheet_draft.xlsx"
coding.to_csv(csv_path, index=False)
try:
    coding.to_excel(xlsx_path, index=False)
    print("Wrote", xlsx_path)
except Exception as e:
    print("Excel write skipped:", e)
print("Wrote", csv_path)
coding.head(30)


## Verification Rule

Rows in `candidate_metric_extractions.csv` are not findings. They are prompts for human verification. A number enters the analysis only after the RA/researcher confirms the source page/table, source type, direct-vs-total job concept, projected-vs-realised status, and whether the denominator is capex, MW, GFA, or duration.
